# BirdCLEF+2026 · HGNetV2-B4 4-fold inference (PyTorch + Gaussian smooth)

Self-contained Kaggle 推理 notebook, 直接吃 4 折 `.pt` 文件 (LSEModel + hgnetv2_b4) 跑 GPU 推理.

**前置**:
1. Add Data: `birdclef-2026` (官方比赛数据);
2. Add Data: 你上传的 hgnetv2_b4 权重数据集, 目录里要有 `best_model_fold0.pt ... fold3.pt`.
3. Internet: OFF (提交时必须关), timm 不会去网络下权重 (我们用 `pretrained=False`).

**关于 TTA**: 之前的 ±2.5s shifted TTA 把每条音频的 forward 次数从 48 涨到 100,
在 hgnetv2_b4 上提交会超时. 这版去掉了 TTA, 改用 `gaussian_filter1d(sigma=0.65)`
做时间维平滑 (和 0.952 项目里 hgnet_branch.py 一致), 推理时间砍掉一半多.

## 1. 路径 + 设备

In [ ]:
import os, gc, math
from pathlib import Path
from time import time

import numpy as np
import pandas as pd
import soundfile
import timm
import torch
import torchaudio
from torch import nn
from torch.nn import functional as F
from torchvision.transforms import v2 as tvt_v2
from tqdm import tqdm

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('torch :', torch.__version__, '| timm:', timm.__version__)

# ------- 比赛数据 ----------------------------------------------------------
DATA = Path('/kaggle/input/birdclef-2026')
TEST_SS = DATA / 'test_soundscapes'
TRAIN_SS = DATA / 'train_soundscapes'

# ------- 4 折权重: 手动写死路径 ---------------------------------------------
# Kaggle 上挂的 hgnetv2_b4 dataset 的实际目录 (里面要有 best_model_fold{0..3}.pt).
MODEL_DIR = Path('/kaggle/input/datasets/czastu/hgnetv2-b4')
print('MODEL_DIR :', MODEL_DIR)
for k in range(4):
    p = MODEL_DIR / f'best_model_fold{k}.pt'
    assert p.exists(), f'missing {p}'
    print(' ', p, f'{p.stat().st_size / 1e6:.1f} MB')


## 2. 配置 (跟训练保持一致)

In [ ]:
RANDOM_SEED   = 1086
SAMPLING_RATE = 32_000
SEGMENT_SEC   = 5
N_CLASSES     = 234
N_FOLDS       = 4

MODEL_NAME    = 'hgnetv2_b4.ssld_stage2_ft_in1k'
HEAD_DROPOUT  = 0.5
LSE_TEMP      = 1.0
DROP_PATH     = 0.0  # 推理用不到

MEL_PARAMS = dict(
    sample_rate=32_000, n_fft=2048, win_length=626, hop_length=313,
    f_min=20, n_mels=256, power=2.0, center=True, pad_mode='reflect',
    norm='slaney', mel_scale='htk',
)
LMS_SHAPE = (256, 256)
TOP_DB    = 80.0

# 推理 batch (GPU). T4 16GB 上 hgnetv2_b4 单 batch 约 0.4GB, 32 已经很稳.
BATCH_SIZE = 32
USE_AMP    = True   # GPU fp16 推理

RANK_AVG   = False  # 折间是否做 rank-normalize 平均

# 时间维 Gaussian 平滑的 sigma (单位: 段, 不是秒). 0 / None 关闭.
# hgnet_branch.py / 0.952 项目默认 0.65, 经验值.
SMOOTH_SIGMA = 0.65

# 调试: 没有 sample_submission (本地 / 试跑) 时, 用 train_soundscapes 的前 N 条.
DEBUG_LOCAL_FILES = 10

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## 3. 模型 + 预处理 (从 `hgnet_train_test` 内联抄过来的最小子集)

In [ ]:
class LSEPooling(nn.Module):
    def __init__(self, pool_axis=-1, temperature=1.0):
        super().__init__()
        self.pool_axis = pool_axis
        self.temperature = temperature
    def forward(self, x):
        return self.temperature * (
            torch.logsumexp(x / self.temperature, axis=self.pool_axis)
            - math.log(x.shape[self.pool_axis])
        )

class LSEHead(nn.Module):
    def __init__(self, num_features, num_classes, dropout=0.2, lse_temperature=1.0):
        super().__init__()
        self.lse_pool = LSEPooling(pool_axis=1, temperature=lse_temperature)
        self.cls_fc = nn.Sequential(
            nn.Linear(num_features, num_features),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(num_features, num_classes),
        )
    def forward(self, h):
        # h: (B, C, F, T) -> mean over F -> (B, C, T) -> (B, T, C)
        h = torch.mean(h, axis=2).transpose(1, 2)
        twl = self.cls_fc(h)
        return self.lse_pool(twl)

class LSEModel(nn.Module):
    def __init__(self, model_name, num_classes=234, head_dropout=0.5,
                 lse_temperature=1.0, drop_path_rate=0.0):
        super().__init__()
        try:
            self.backbone = timm.create_model(
                model_name, pretrained=False, in_chans=1,
                global_pool='', num_classes=0, drop_path_rate=drop_path_rate,
            )
        except TypeError:
            self.backbone = timm.create_model(
                model_name, pretrained=False, in_chans=1,
                global_pool='', num_classes=0,
            )
        with torch.no_grad():
            num_features = self.backbone(torch.randn(1, 1, 256, 256)).shape[1]
        self.head = LSEHead(num_features, num_classes, head_dropout, lse_temperature)
    def forward(self, x):
        return self.head(self.backbone(x))


class LogMelSpectrogramTransform(nn.Module):
    def __init__(self, mel_params, top_db, lms_shape=(256, 256)):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(**mel_params)
        self.db  = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=top_db)
        self.resize = tvt_v2.Resize(size=lms_shape)
    @torch.no_grad()
    def forward(self, wave):
        lms = self.db(self.mel(wave))           # (B, n_mels, T)
        lms = self.resize(lms)                  # (B, H, W)
        b = lms.shape[0]
        flat = lms.reshape(b, -1)
        mn = flat.min(dim=1)[0][:, None, None]
        mx = flat.max(dim=1)[0][:, None, None]
        lms = (lms - mn) / (mx - mn + 1e-7)
        return lms[:, None, :, :]               # (B, 1, H, W)


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

def rank_normalize(x):
    out = np.zeros_like(x)
    for i in range(x.shape[1]):
        r = pd.Series(x[:, i]).rank(method='max').values
        out[:, i] = r / r.shape[0]
    return out


## 4. 测试集路径 + row_id

In [ ]:
sample_sub = pd.read_csv(DATA / 'sample_submission.csv')
# 判断逻辑改成 '看 test_soundscapes 目录里是不是真的有 ogg 文件',
# 这样 Kaggle commit 阶段 (sample_submission 只有几行预览) 和 submit 阶段 (真实测试集)
# 都能正确走 test 分支, 避免落到 debug fallback 上导致全部 NaN.
test_ogg_files = sorted(TEST_SS.glob('*.ogg')) if TEST_SS.exists() else []
is_test_env = len(test_ogg_files) > 0
print('sample_submission rows =', len(sample_sub),
      '| #test_soundscapes ogg =', len(test_ogg_files),
      '| is_test_env =', is_test_env)

if is_test_env:
    # 用 sample_submission 里出现的 file_id 来确定推理文件列表
    # (row_id 形如 '<file_id>_<end_sec>', 取前缀去重).
    test_ss_paths, added = [], set()
    for row_id in sample_sub['row_id'].values:
        file_id = '_'.join(row_id.split('_')[:-1])
        if file_id in added:
            continue
        added.add(file_id)
        test_ss_paths.append(TEST_SS / f'{file_id}.ogg')
    # 兜底: sample_submission 行数太少 (commit 预览阶段) 时,
    # 把目录里所有 ogg 都跑一遍, 避免漏文件影响 commit 检查.
    if len(test_ss_paths) < len(test_ogg_files):
        seen = {p.name for p in test_ss_paths}
        for p in test_ogg_files:
            if p.name not in seen:
                test_ss_paths.append(p)
else:
    # 本地 / 没有 test_soundscapes 时 debug 用.
    test_ss_paths = sorted(TRAIN_SS.iterdir())[:DEBUG_LOCAL_FILES]
print('#test files =', len(test_ss_paths))

# row_id 列表: file_stem_(5/10/.../60)
test_ss_segs = []
for p in test_ss_paths:
    for end_sec in range(5, 65, 5):
        test_ss_segs.append(f'{p.stem}_{end_sec}')

taxonomy = pd.read_csv(DATA / 'taxonomy.csv')
CLASSES = taxonomy.primary_label.values.tolist()
assert len(CLASSES) == N_CLASSES


## 5. 切片: 每条 60s 切 12 段 (无 TTA)

In [ ]:
max_sec  = 60
duration = SAMPLING_RATE * max_sec
seg_len  = 5 * SAMPLING_RATE

normal_list = []
for path in tqdm(test_ss_paths, desc='loading test waves'):
    with soundfile.SoundFile(path) as f:
        n_frames = f.frames
        wave = np.zeros(duration, dtype='float32')
        if n_frames < duration:
            wave[:n_frames] = f.read(dtype='float32')
        else:
            wave[:] = f.read(frames=duration, dtype='float32')
    # 12 段 5s: [0..5), [5..10), ..., [55..60)
    for s in range(0, max_sec, 5):
        normal_list.append(wave[s * SAMPLING_RATE : (s + 5) * SAMPLING_RATE])

num_audios = len(test_ss_paths)
num_normal = len(normal_list)
print(f'normal segs: {num_normal} ({num_normal // 12} files * 12)')

normal_waves = np.stack(normal_list, axis=0).astype(np.float32)
del normal_list
gc.collect()
print('normal_waves :', normal_waves.shape)


## 6. 4 折 PyTorch 推理

In [ ]:
lms_transform = LogMelSpectrogramTransform(MEL_PARAMS, TOP_DB, LMS_SHAPE).eval().to(device)

@torch.no_grad()
def run_inference(model, waves: np.ndarray, batch_size: int) -> np.ndarray:
    n = waves.shape[0]
    out = np.zeros((n, N_CLASSES), dtype=np.float32)
    for i in tqdm(range(0, n, batch_size), desc='infer', leave=False):
        wb = torch.from_numpy(waves[i : i + batch_size]).to(device, non_blocking=True)
        lms = lms_transform(wb)
        if USE_AMP and device.type == 'cuda':
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(lms)
        else:
            logits = model(lms)
        out[i : i + batch_size] = logits.float().cpu().numpy()
    return sigmoid_np(out)

pred_normal  = np.zeros((N_FOLDS, num_normal,  N_CLASSES), dtype=np.float32)

for fold_id in range(N_FOLDS):
    ck_path = MODEL_DIR / f'best_model_fold{fold_id}.pt'
    print(f'\n[fold {fold_id}] loading {ck_path.name}')
    model = LSEModel(
        MODEL_NAME, num_classes=N_CLASSES,
        head_dropout=HEAD_DROPOUT, lse_temperature=LSE_TEMP,
        drop_path_rate=DROP_PATH,
    )
    state = torch.load(ck_path, map_location='cpu', weights_only=False)
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        print('  missing keys   :', missing[:5], f'... ({len(missing)})')
    if unexpected:
        print('  unexpected keys:', unexpected[:5], f'... ({len(unexpected)})')
    model.eval().to(device)

    t0 = time()
    pred_normal[fold_id] = run_inference(model, normal_waves, BATCH_SIZE)
    print(f'  normal done in {time() - t0:.1f}s')

    del model, state
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()


## 7. 折间平均 + Gaussian 时间平滑 + 写 `submission.csv`

In [ ]:
from scipy.ndimage import gaussian_filter1d

# pred_normal: (N_FOLDS, num_normal, N_CLASSES) -> (N_FOLDS, num_audios, 12, N_CLASSES)
pn = pred_normal.reshape(N_FOLDS, num_audios, 12, N_CLASSES)

# 1) 折间融合
if RANK_AVG:
    pn_flat = pn.reshape(N_FOLDS, num_normal, N_CLASSES)
    rank = np.zeros_like(pn_flat)
    for k in range(N_FOLDS):
        rank[k] = rank_normalize(pn_flat[k])
    avg = rank.mean(axis=0).reshape(num_audios, 12, N_CLASSES)
else:
    avg = pn.mean(axis=0)  # (num_audios, 12, N_CLASSES)

# 2) 时间维 Gaussian 平滑 (每条音频独立, 12 段之间)
if SMOOTH_SIGMA and SMOOTH_SIGMA > 0:
    avg = gaussian_filter1d(avg, sigma=SMOOTH_SIGMA, axis=1, mode='nearest').astype(np.float32)

avg = np.clip(avg, 0.0, 1.0).reshape(num_normal, N_CLASSES)

sub_df = pd.DataFrame(avg, columns=CLASSES,
                      index=pd.Series(test_ss_segs, name='row_id')).reset_index()
# 真实提交时按 sample_submission 的 row_id 顺序对齐;
# commit 预览阶段 sample_sub 行数远小于推理结果, 这时直接保留全部预测.
if len(sample_sub) >= len(sub_df):
    sub_df = pd.merge(sample_sub[['row_id']], sub_df, on='row_id', how='left')
sub_df.to_csv('submission.csv', index=False)
print('submission.csv shape =', sub_df.shape)
sub_df.head()
